# 2.1 — Spark Connect

**Chapter 2, section 2.2.4** (*Spark Connect*).

**The question this notebook answers:** if the driver moves to the server, what is left on the
client — and what stops working?

Until recently, running a PySpark program required running a JVM. Even in local mode the
Python process launched one, and the driver lived inside it. Spark Connect splits that apart:
the client becomes a thin library that builds an **unresolved logical plan** and sends it over
gRPC to a server, which owns the driver and does the work.

The consequence this notebook makes concrete is the one the chapter states and then relies on
for the rest of the chapter:

> Spark Connect transmits a logical plan, so it can only carry operations that are expressible
> as one. The structured, plan-based DataFrame API satisfies this condition; the lower-level
> interface introduced in the next section, in which arbitrary Python functions are applied
> directly to the JVM's own objects, does not.

So this notebook is the only one in chapter 2 that does **not** use `sc`. Everything after it
does, and section 4 below is why.

Runs on a laptop in under a minute.

## 1. The conventional arrangement, for comparison

This is what every other notebook in this chapter uses: one process, one JVM, the driver
inside it, and a `SparkContext` reachable from the session.

In [1]:
import os, tempfile
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

classic = (SparkSession.builder
           .appName("CS777-2.1-classic")
           .master("local[*]")
           .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
           .config("spark.ui.showConsoleProgress", "false")
           .getOrCreate())
classic.sparkContext.setLogLevel("ERROR")

print("session class :", type(classic).__module__)
print("version       :", classic.version)
print("sparkContext  :", classic.sparkContext.master, "  <- reachable")
print("an RDD        :", classic.sparkContext.parallelize([1, 2, 3]).sum())

session class : pyspark.sql.session
version       : 4.2.0
sparkContext  : local[*]   <- reachable


an RDD        : 6


In [2]:
# Stop it before opening a Connect session, so that the two arrangements do not overlap
# and it is unambiguous which one each result below came from.
classic.stop()
print("classic session stopped")

classic session stopped


## 2. A Connect session

`SparkSession.builder.remote(...)` replaces `.master(...)`. The two are alternatives: `master`
says *start a driver here and attach it to that cluster manager*; `remote` says *there is
already a driver somewhere, talk to it*.

Passing `remote("local[*]")` is the development shortcut. It starts a Connect **server** on
this machine for you and points the client at it, which is the arrangement used below so that
the notebook runs anywhere. It is worth being honest about what that does and does not
demonstrate: the client–server split is real and every operation below genuinely crosses gRPC,
but the JVM has not gone anywhere — it is running as the server, on this machine. Section 5
shows the form that puts it on a different machine.

In [3]:
spark = (SparkSession.builder
         .appName("CS777-2.1-connect")
         .remote("local[*]")
         .getOrCreate())

print("session class :", type(spark).__module__, " <- note: .connect.")
print("version       :", spark.version)
print("shuffle parts :", spark.conf.get("spark.sql.shuffle.partitions"))
print("client        :", type(spark.client).__name__)
print("session id    :", spark.client._session_id)

session class : pyspark.sql.connect.session  <- note: .connect.
version       : 4.2.0
shuffle parts : 200
client        : SparkConnectClient
session id    : b6fed12d-52aa-4221-9d37-cda34637d222


## 3. What works: anything expressible as a plan

The whole structured API is available, because every operation in it becomes a node in a
logical plan, and a logical plan is exactly what gRPC carries.

In [4]:
df = spark.createDataFrame(
    [("Chris", "Espresso", 5), ("Peter", "Latte", 9), ("John", "Cold Brew", 6),
     ("Chris", "Latte", 7)],
    "taster string, coffee string, score int")

print("DataFrame class:", type(df).__module__, "\n")
df.show()

print("a filter and an aggregation, both planned on the server:")
(df.filter(F.col("score") > 5)
   .groupBy("taster")
   .agg(F.avg("score").alias("avg_score"), F.count("*").alias("n"))
   .orderBy("taster")
   .show())

DataFrame class: pyspark.sql.connect.dataframe 



+------+---------+-----+
|taster|   coffee|score|
+------+---------+-----+
| Chris| Espresso|    5|
| Peter|    Latte|    9|
|  John|Cold Brew|    6|
| Chris|    Latte|    7|
+------+---------+-----+

a filter and an aggregation, both planned on the server:


+------+---------+---+
|taster|avg_score|  n|
+------+---------+---+
| Chris|      7.0|  1|
|  John|      6.0|  1|
| Peter|      9.0|  1|
+------+---------+---+



In [5]:
# SQL works too: it is parsed into the same logical plan.
df.createOrReplaceTempView("drinks")
spark.sql("SELECT coffee, max(score) AS best FROM drinks GROUP BY coffee ORDER BY coffee").show()

# And so does explain(), which is a good demonstration of where the work happens: the
# client never planned anything.  It sent an unresolved plan and asked the server to
# describe what it made of it.
df.filter(F.col("score") > 5).select("taster", "score").explain()

+---------+----+
|   coffee|best|
+---------+----+
|Cold Brew|   6|
| Espresso|   5|
|    Latte|   9|
+---------+----+

== Physical Plan ==
LocalTableScan [taster#107, score#109]




In [6]:
# Reading and writing files works as well -- but note WHOSE file system is meant.  The
# server opens the path, not the client.  In local mode they are the same machine, which
# hides the distinction; against a remote server, a local path on your laptop is not
# visible to it, and the path must be one the server can reach (object storage, usually).
out = os.path.join(SCRATCH, "ch02-connect-demo.parquet")
df.write.mode("overwrite").parquet(out)
print("rows read back:", spark.read.parquet(out).count())

rows read back: 4


## 4. What does not work: the RDD API

This is the limitation the chapter records before the rest of chapter 2 relies on the older
interface, and it is not an accident of the implementation. An RDD holds a **Python function
applied to the JVM's own objects**. There is no way to express "run this arbitrary Python
lambda over the partitions" as a node in a logical plan, so there is nothing for gRPC to send.

Both failures below are caught deliberately: this cell is the point of the notebook, not an
error in it.

In [7]:
print("Attempting the operations chapter 2 uses everywhere else:\n")

try:
    sc = spark.sparkContext
    print("spark.sparkContext ->", sc)
except Exception as e:
    print(f"spark.sparkContext -> {type(e).__name__}")
    print(f"    {str(e).splitlines()[0]}")

print()

try:
    rows = df.rdd.take(1)
    print("df.rdd ->", rows)
except Exception as e:
    print(f"df.rdd             -> {type(e).__name__}")
    print(f"    {str(e).splitlines()[0]}")

print("\nBoth messages say the same thing in two ways: the operation needs the JVM's own")
print("objects, and this session does not have a JVM -- it has a socket to one.")

Attempting the operations chapter 2 uses everywhere else:

spark.sparkContext -> PySparkAttributeError
    [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `sparkContext` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.

df.rdd             -> PySparkNotImplementedError
    [NOT_IMPLEMENTED] rdd is not implemented.

Both messages say the same thing in two ways: the operation needs the JVM's own
objects, and this session does not have a JVM -- it has a socket to one.


In [8]:
# The same computation, done the way a Connect session can express it.  This is the
# migration path for RDD code in general: say WHAT is wanted, not HOW to walk the records.
print("RDD form (needs sc)      : rdd.map(lambda r: r.score * 2).sum()")
print("Plan form (works here)   : df.select(F.sum(F.col('score') * 2))\n")
df.select(F.sum(F.col("score") * 2).alias("doubled_total")).show()

RDD form (needs sc)      : rdd.map(lambda r: r.score * 2).sum()
Plan form (works here)   : df.select(F.sum(F.col('score') * 2))

+-------------+
|doubled_total|
+-------------+
|           54|
+-------------+



## 5. A server on another machine

The form above starts the server for you. The real arrangement starts it once, separately, and
points any number of thin clients at it. On a machine with a Spark distribution installed:

```bash
# On the server:
$SPARK_HOME/sbin/start-connect-server.sh \
    --packages org.apache.spark:spark-connect_2.13:4.2.0
# listens on port 15002 by default
```

```python
# On the client -- no Java installed, no JVM started, nothing but the pyspark client library:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://spark-host:15002").getOrCreate()
```

Three consequences follow from the split, and they are the reason it exists:

* **Several users share one cluster** without each spawning a driver.
* **A crash in the client no longer kills the Spark application.** The driver is on the server;
  the client is a socket.
* **The server can be upgraded without every client upgrading at the same time**, because what
  crosses the wire is a plan, not a serialized JVM object graph.

In [9]:
spark.stop()
print("connect session stopped")

connect session stopped


## Conclusion

| | Classic session | Connect session |
|---|---|---|
| Created with | `.master("local[*]")` | `.remote("sc://host:15002")` |
| Where the driver runs | in the client's own JVM | on the server |
| Client needs Java | yes | no |
| Class | `pyspark.sql.session` | `pyspark.sql.connect.session` |
| DataFrame API, SQL | yes | yes |
| `spark.sparkContext` | yes | **no** — `JVM_ATTRIBUTE_NOT_SUPPORTED` |
| `df.rdd`, `sc.parallelize` | yes | **no** — `NOT_IMPLEMENTED` |

The practical reading is the chapter's: **a Connect client is the modern way to reach a cluster
for structured work**, and the material that follows in this chapter — RDDs, `sc`, arbitrary
Python functions over partitions — is run in the conventional arrangement, because it cannot be
expressed as a plan and therefore cannot be sent.

That is worth holding onto beyond this notebook. It is the same boundary that appears in
section 2.2.5 on native execution engines: an operation the engine can express in its own terms
runs on the fast path, and a Python function applied to raw records does not. Spark Connect
draws the line at what can be *transmitted*; a native engine draws it at what can be
*accelerated*. It is the same line.

**Next.** Notebook 2.2 is the reference tour of the RDD API, in a conventional session.